# Unified Multi-Sensor SPIReS Playground

This notebook exercises the unified `spires.sensors` API across all five currently wired sensor/platform combinations in this repo:

- MODIS Terra
- MODIS Aqua
- VIIRS SNPP
- VIIRS NOAA-20
- VIIRS NOAA-21

For each platform it will:

- build or reuse an `R0` background from the local summer stack
- run inversion on every scene found in the local `reflectance/` directory
- save a netCDF result for each scene
- save and display a 1x4 quicklook figure for each scene

The notebook assumes the local data layout currently present under `data/modis/inputs/` and `data/viirs/inputs/`.


In [ ]:
from pathlib import Path
import sys
from time import perf_counter

from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

search_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(path for path in search_roots if (path / 'spires').exists() and (path / 'tests' / 'data').exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from spires import configure_spires_file_logger, make_spires_log_path
from spires.sensors import (
    build_r0_from_sources,
    load_output_dataset_if_valid,
    prepare_scene_for_inversion,
    run_inversion,
    validate_inversion_output_dataset,
    write_output_dataset,
)

PLATFORM_LABELS = {
    ('modis', 'terra'): 'MODIS Terra',
    ('modis', 'aqua'): 'MODIS Aqua',
    ('viirs', 'snpp'): 'VIIRS SNPP',
    ('viirs', 'noaa20'): 'VIIRS NOAA-20',
    ('viirs', 'noaa21'): 'VIIRS NOAA-21',
}

RUN_CONFIGS = [
    {
        'sensor': 'modis',
        'platform': 'terra',
        'input_root': repo_root / 'data' / 'modis' / 'inputs' / 'terra',
        'reflectance_glob': '*.hdf',
        'r0_glob': '**/*.hdf',
        'lut_path': repo_root / 'data' / 'modis' / 'lut' / 'lut_modis_1_2_3_4_5_6_7_3um_dust_bandpass.mat',
        'cloud_mask_policy': 'ignore_cloud_and_shadow',
    },
    {
        'sensor': 'modis',
        'platform': 'aqua',
        'input_root': repo_root / 'data' / 'modis' / 'inputs' / 'aqua',
        'reflectance_glob': '*.hdf',
        'r0_glob': '**/*.hdf',
        'lut_path': repo_root / 'data' / 'modis' / 'lut' / 'lut_modis_1_2_3_4_5_6_7_3um_dust_bandpass.mat',
        'cloud_mask_policy': 'ignore_cloud_and_shadow',
    },
    {
        'sensor': 'viirs',
        'platform': 'snpp',
        'input_root': repo_root / 'data' / 'viirs' / 'inputs' / 'snpp',
        'reflectance_glob': '*.h5',
        'r0_glob': '**/*.h5',
        'lut_path': repo_root / 'data' / 'viirs' / 'lut' / 'lut_viirs_snpp_i1_i2_i3_m2_m4_m8_m11_3um_dust_bandpass.mat',
        'cloud_mask_policy': 'ignore_cloud_and_shadow',
    },
    {
        'sensor': 'viirs',
        'platform': 'noaa20',
        'input_root': repo_root / 'data' / 'viirs' / 'inputs' / 'noaa20',
        'reflectance_glob': '*.h5',
        'r0_glob': '**/*.h5',
        'lut_path': repo_root / 'data' / 'viirs' / 'lut' / 'lut_viirs_noaa20_i1_i2_i3_m2_m4_m8_m11_3um_dust_bandpass.mat',
        'cloud_mask_policy': 'ignore_cloud_and_shadow',
    },
    {
        'sensor': 'viirs',
        'platform': 'noaa21',
        'input_root': repo_root / 'data' / 'viirs' / 'inputs' / 'noaa21',
        'reflectance_glob': '*.h5',
        'r0_glob': '**/*.h5',
        'lut_path': repo_root / 'data' / 'viirs' / 'lut' / 'lut_viirs_noaa21_i1_i2_i3_m2_m4_m8_m11_3um_dust_bandpass.mat',
        'cloud_mask_policy': 'ignore_cloud_and_shadow',
    },
]

ZARR_CHUNKS = {'time': 1, 'y': 512, 'x': 512, 'band': -1}
EXECUTION_PROFILE = 'local'
APPLY_VALID_INVERSION_MASK = False
REUSE_EXISTING_INVERSION_OUTPUTS = False


In [ ]:
def gather_run_info(config):
    sensor = config['sensor']
    platform = config['platform']
    input_root = config['input_root']
    reflectance_dir = input_root / 'reflectance'
    r0_input_dir = input_root / 'r0'
    reflectance_paths = sorted(reflectance_dir.glob(config['reflectance_glob']))
    r0_source_paths = sorted(r0_input_dir.glob(config['r0_glob']))
    reference_path = reflectance_paths[0] if reflectance_paths else (r0_source_paths[0] if r0_source_paths else None)
    if reference_path is None:
        raise FileNotFoundError(f'No source scenes found for {sensor}/{platform} under {input_root}')

    if sensor == 'modis':
        data_root = repo_root / 'data' / 'modis'
    else:
        data_root = repo_root / 'data' / 'viirs'

    parts = reference_path.name.split('.')
    tile = parts[2] if len(parts) > 2 else 'unknown_tile'
    reflectance_year = int(parts[1][1:5]) if len(parts) > 1 and parts[1].startswith('A') else 2023
    r0_year = int(r0_source_paths[0].name.split('.')[1][1:5]) if r0_source_paths else reflectance_year

    if sensor == 'modis':
        r0_path = data_root / 'r0' / platform / tile / str(r0_year) / f'modis_r0_{platform}_{tile}_{r0_year}.nc'
    else:
        r0_path = data_root / 'r0' / platform / tile / str(r0_year) / f'viirs_r0_{platform}_{tile}_{r0_year}.nc'
    r0_path.parent.mkdir(parents=True, exist_ok=True)

    output_root = repo_root / 'outputs' / 'unified_multisensor' / sensor / platform / tile / str(reflectance_year)
    figure_dir = output_root / 'figures'
    netcdf_dir = output_root / 'netcdf'
    timeseries_dir = output_root / 'timeseries'
    logs_dir = output_root / 'logs'
    figure_dir.mkdir(parents=True, exist_ok=True)
    netcdf_dir.mkdir(parents=True, exist_ok=True)
    timeseries_dir.mkdir(parents=True, exist_ok=True)
    logs_dir.mkdir(parents=True, exist_ok=True)

    zarr_path = timeseries_dir / f'{sensor}_{platform}_{tile}_{r0_year}_r0_stack.zarr'
    log_path = make_spires_log_path(logs_dir, prefix=f'unified_{sensor}_{platform}_{tile}_{reflectance_year}')

    return {
        **config,
        'tile': tile,
        'reflectance_year': reflectance_year,
        'r0_year': r0_year,
        'reflectance_dir': reflectance_dir,
        'r0_input_dir': r0_input_dir,
        'reflectance_paths': reflectance_paths,
        'r0_source_paths': r0_source_paths,
        'r0_path': r0_path,
        'figure_dir': figure_dir,
        'netcdf_dir': netcdf_dir,
        'timeseries_dir': timeseries_dir,
        'zarr_path': zarr_path,
        'log_path': log_path,
        'label': PLATFORM_LABELS[(sensor, platform)],
    }


def print_run_summary(run):
    lines = [
        f"Label: {run['label']}",
        f"Input root: {run['input_root']}",
        f"R0 input dir: {run['r0_input_dir']}",
        f"R0 source scenes found: {len(run['r0_source_paths'])}",
        f"Reflectance input dir: {run['reflectance_dir']}",
        f"Reflectance scenes found: {len(run['reflectance_paths'])}",
        f"Tile: {run['tile']}",
        f"Reflectance year: {run['reflectance_year']}",
        f"R0 year: {run['r0_year']}",
        f"LUT: {run['lut_path']}",
        f"R0 output: {run['r0_path']}",
        f"Output figures: {run['figure_dir']}",
        f"Output netCDF: {run['netcdf_dir']}",
        f"Timeseries zarr: {run['zarr_path']}",
        f"Log path: {run['log_path']}",
        f"Cloud mask policy: {run['cloud_mask_policy']}",
    ]
    for line in lines:
        print(line)


runs = [gather_run_info(config) for config in RUN_CONFIGS]
for run in runs:
    print('=' * 100)
    print_run_summary(run)


## Unified 1x4 Quicklooks

The plot function below is sensor-generic and expects the common inversion outputs returned by the unified API.


In [ ]:
def plot_inversion_1x4(inversion_ds, *, label, acquisition_date, figure_path):
    fig, ax = plt.subplots(1, 4, figsize=(24, 5))
    (inversion_ds.raw_viewable_snow_fraction * 100).plot(ax=ax[0], cmap='YlGnBu_r', vmin=0, vmax=100)
    (inversion_ds.raw_shade_fraction * 100).plot(ax=ax[1], cmap='Greys', vmin=0, vmax=100)
    (inversion_ds.raw_canopy_fraction * 100).plot(ax=ax[2], cmap='YlGn', vmin=0, vmax=100)
    (inversion_ds.raw_snow_fraction * 100).plot(ax=ax[3], cmap='YlGnBu_r', vmin=0, vmax=100)

    ax[0].set_title('SpiPy raw viewable snow fraction')
    ax[1].set_title('SpiPy raw shade fraction')
    ax[2].set_title('SpiPy raw canopy fraction')
    ax[3].set_title('SpiPy raw snow fraction')
    fig.suptitle(f'{label} // {acquisition_date}', y=1.02)
    fig.tight_layout()
    fig.savefig(figure_path, dpi=300, bbox_inches='tight')
    plt.show()
    return fig


## Batch Run

This cell builds or reuses `R0` for each platform, then runs inversion on every reflectance scene found locally. The same unified API is used for all five platforms.


In [ ]:
all_outputs = []
overall_start = perf_counter()

for run in runs:
    sensor = run['sensor']
    platform = run['platform']
    logger = configure_spires_file_logger(
        run['log_path'],
        logger_name=f'spires.examples.unified.{sensor}.{platform}.{run["tile"]}.{run["reflectance_year"]}',
        log_to_stdout=False,
    )

    print(f"\n{'#' * 100}")
    print(f"Running {run['label']}")
    print(f"{'#' * 100}")

    r0_build_start = perf_counter()
    r0_ds = build_r0_from_sources(
        run['r0_source_paths'],
        sensor=sensor,
        platform=platform,
        r0_path=run['r0_path'],
        lut_file=run['lut_path'],
        logger=logger,
        show_progress=True,
        cloud_mask_policy=run['cloud_mask_policy'],
        zarr_path=run['zarr_path'],
        chunks=ZARR_CHUNKS,
    )
    print(f"R0 ready in {perf_counter() - r0_build_start:.2f} s -> {run['r0_path']}")

    for scene_path in run['reflectance_paths']:
        print(f'Processing {scene_path.name} ...')
        scene_start = perf_counter()

        scene_ds = prepare_scene_for_inversion(
            scene_path,
            sensor=sensor,
            platform=platform,
            lut_file=run['lut_path'],
            logger=logger,
            cloud_mask_policy=run['cloud_mask_policy'],
        )
        acquisition_date = scene_ds.attrs.get('acquisition_date', 'unknown-date')
        tile = scene_ds.attrs.get('tile', run['tile'])
        product = scene_ds.attrs.get('product', f'{sensor}_{platform}')
        date_slug = acquisition_date.replace('-', '')
        stem = f'{product}_{platform}_{tile}_{date_slug}'
        figure_path = run['figure_dir'] / f'{stem}_quicklook.png'
        netcdf_path = run['netcdf_dir'] / f'{stem}_inversion.nc'

        expected_output_attrs = {
            'product': product,
            'platform': platform,
            'tile': tile,
            'acquisition_date': acquisition_date,
        }

        inversion_ds = None
        reused_inversion_output = False
        if REUSE_EXISTING_INVERSION_OUTPUTS:
            inversion_ds = load_output_dataset_if_valid(
                netcdf_path,
                validator=validate_inversion_output_dataset,
                expected_attrs=expected_output_attrs,
            )
            if inversion_ds is not None:
                reused_inversion_output = True
                print(f'  reused netCDF -> {netcdf_path.name}')

        if inversion_ds is None:
            lazy_inversion = run_inversion(
                scene_ds,
                r0_ds,
                sensor=sensor,
                platform=platform,
                lut_file=run['lut_path'],
                execution_profile=EXECUTION_PROFILE,
                logger=logger,
                apply_valid_inversion_mask=APPLY_VALID_INVERSION_MASK,
            )
            with ProgressBar(dt=5.0):
                inversion_ds = lazy_inversion.compute()

            write_output_dataset(
                inversion_ds,
                netcdf_path,
                validator=validate_inversion_output_dataset,
                expected_attrs=expected_output_attrs,
            )
            print(f'  saved netCDF -> {netcdf_path.name}')

        rendered_figure = False
        if (not figure_path.exists()) or (not reused_inversion_output):
            plot_inversion_1x4(
                inversion_ds,
                label=run['label'],
                acquisition_date=acquisition_date,
                figure_path=figure_path,
            )
            rendered_figure = True

        elapsed_seconds = perf_counter() - scene_start
        all_outputs.append(
            {
                'sensor': sensor,
                'platform': platform,
                'label': run['label'],
                'scene': scene_path.name,
                'acquisition_date': acquisition_date,
                'tile': tile,
                'netcdf': str(netcdf_path),
                'figure': str(figure_path),
                'elapsed_seconds': elapsed_seconds,
            }
        )
        if rendered_figure:
            print(f'  saved figure -> {figure_path.name}')
        else:
            print(f'  reused figure -> {figure_path.name}')
        print(f'  elapsed      -> {elapsed_seconds:.2f} s')

print(f'\nProcessed {len(all_outputs)} scene(s) in {(perf_counter() - overall_start) / 60:.2f} minutes')
results_df = pd.DataFrame(all_outputs)
results_df


## Output Summary

Use this cell to quickly review the saved netCDF and quicklook paths after the batch finishes.


In [ ]:
if 'results_df' in globals() and not results_df.empty:
    display(results_df[['label', 'scene', 'acquisition_date', 'netcdf', 'figure', 'elapsed_seconds']])
else:
    print('No outputs have been generated yet.')


In [ ]:
inversion_ds